<a href="https://colab.research.google.com/github/zune2/DSpython/blob/main/2026-10-05-EP002-faiss-local-rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TF-IDF + FAISS 결합 로컬 RAG 검색 코드

In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 28.8 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import faiss
import warnings

warnings.filterwarnings(action='ignore')

# =====================================================================
# [RAG Step 1] 지식 베이스 구축 (사내 업무 매뉴얼 데이터)
# =====================================================================
company_knowledge = [
    {"doc_id": 0, "topic": "연차 휴가 신청", "content": "연차 휴가는 사용 예정일 최소 3일 전까지 사내 그룹웨어를 통해 신청서를 제출해야 하며, 팀장 및 부서장의 승인이 필요합니다."},
    {"doc_id": 1, "topic": "재택 근무 규정", "content": "재택 근무는 주 2회까지 신청 가능하며, 전날 퇴근 전까지 재택 근무 신청 메뉴에서 사유를 적어 결재를 올려야 합니다."},
    {"doc_id": 2, "topic": "법인 카드 사용", "content": "법인 카드는 식대 지원 한도가 1인당 15,000원이며, 주말 사용 시에는 반드시 증빙 서류와 사유서를 첨부하여 재경팀에 제출해야 합니다."},
    {"doc_id": 3, "topic": "복지 포인트 혜택", "content": "매년 1월 1일 전 임직원에게 복지 포인트 100만 포인트가 지급되며, 사내 복지몰 또는 자기개발, 도서 구매, 건강검진 비용으로 사용할 수 있습니다."},
    {"doc_id": 4, "topic": "출장비 정산 가이드", "content": "국내 출장 시 KTX 및 고속버스 영수증을 첨부하면 실비 정산되며, 일비는 하루 30,000원, 숙박비는 1박 기준 최대 70,000원까지 지원됩니다."}
]

df_knowledge = pd.DataFrame(company_knowledge)

# =====================================================================
# [RAG Step 2] TF-IDF 변환 및 FAISS 규격 맞춤 가공 (float32)
# =====================================================================
# 한국어 조사를 깨부수는 글자 n-gram 분석기 정의
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4))
tfidf_matrix = vectorizer.fit_transform(df_knowledge['content'])

# ⚠️ FAISS 연동 핵심 치트키 1:
# scikit-learn의 희소 행렬을 FAISS가 요구하는 일반 넘파이 배열(float32)로 강제 변환합니다.
database_vectors = tfidf_matrix.toarray().astype('float32')

# ⚠️ FAISS 연동 핵심 치트키 2 (코사인 유사도 매칭 셋팅):
# 벡터들의 크기를 1로 똑같이 맞춰주는 정규화(L2 Normalization)를 수행하면,
# FAISS의 내적(Inner Product) 인덱스가 완벽한 '코사인 유사도 계산기'로 변신합니다.
faiss.normalize_L2(database_vectors)

# =====================================================================
# [RAG Step 3] FAISS 인덱스 창고 생성 및 데이터 주입
# =====================================================================
# 벡터의 차원 수 추출 (TF-IDF가 생성한 단어 조합 가중치 개수)
dimension = database_vectors.shape[1]

# IndexFlatIP: 내적(Inner Product) 기반 인덱스 생성 (정규화된 벡터 입력 시 코사인 유사도와 동일)
faiss_index = faiss.IndexFlatIP(dimension)

# 생성된 인덱스 창고에 매뉴얼 벡터들 때려 넣기
faiss_index.add(database_vectors)

print("=== 📦 FAISS 기반 인메모리 벡터 DB 구축 완료 ===")
print(f"👉 총 {faiss_index.ntotal}개의 매뉴얼 문서가 {dimension}차원 공간에 초고속 인덱싱되었습니다.\n")

# =====================================================================
# [RAG Step 4] FAISS 기반 초고속 검색 함수 정의 (Retrieval)
# =====================================================================
def faiss_rag_retriever(user_query, top_k=1):
    print(f"👤 사용자 질문: '{user_query}'")

    # 1. 질문을 매뉴얼 데이터와 똑같은 규격의 float32 넘파이 벡터로 변환
    query_vector = vectorizer.transform([user_query]).toarray().astype('float32')

    # 2. 질문 벡터 역시 L2 정규화를 거쳐 각도 계산 대기 상태로 만듦
    faiss.normalize_L2(query_vector)

    # 3. FAISS 창고 안에서 질문과 코사인 유사도가 가장 높은 상위 문서 탑 K개 초고속 서칭
    # 유사도 점수(similarities)와 매칭된 문서 인덱스 번호(indices)가 반환됩니다.
    similarities, indices = faiss_index.search(query_vector, top_k)

    best_doc_idx = indices[0][0]
    best_score = similarities[0][0]

    print(f"🚀 [FAISS 검색 엔진] 연관 지식 초고속 스캔 완료 (최고 코사인 유사도: {best_score:.4f})")
    print("-" * 65)

    # 신뢰성 방어선 (엉뚱한 질문 차단용 환각 방지선)
    if best_score < 0.10:
        return "❌ 죄송합니다. 질문하신 내용과 관련된 사내 매뉴얼 지식을 찾을 수 없습니다."

    # 4. 고유 인덱스 번호로 매뉴얼 지식 추출
    retrieved_knowledge = df_knowledge.iloc[best_doc_idx]

    output_response = f"📌 매칭 매뉴얼: 【{retrieved_knowledge['topic']}】 (유사도: {best_score:.4f})\n💡 AI 핵심 지식 컨텍스트 제공:\n{retrieved_knowledge['content']}"
    return output_response

# =====================================================================
# [RAG Step 5] 실무 가동 테스트
# =====================================================================
# 구어체 질문 테스트 1
print(faiss_rag_retriever("기차 타고 국내 출장 갈 때 숙박비 영수증 정산 어떻게 해?"))
print("\n" + "="*65 + "\n")

# 구어체 질문 테스트 2
print(faiss_rag_retriever("복지 포인트 백만 원 나온 거 어디서 쓸 수 있어?"))


=== 📦 FAISS 기반 인메모리 벡터 DB 구축 완료 ===
👉 총 5개의 매뉴얼 문서가 921차원 공간에 초고속 인덱싱되었습니다.

👤 사용자 질문: '기차 타고 국내 출장 갈 때 숙박비 영수증 정산 어떻게 해?'
🚀 [FAISS 검색 엔진] 연관 지식 초고속 스캔 완료 (최고 코사인 유사도: 0.3199)
-----------------------------------------------------------------
📌 매칭 매뉴얼: 【출장비 정산 가이드】 (유사도: 0.3199)
💡 AI 핵심 지식 컨텍스트 제공:
국내 출장 시 KTX 및 고속버스 영수증을 첨부하면 실비 정산되며, 일비는 하루 30,000원, 숙박비는 1박 기준 최대 70,000원까지 지원됩니다.


👤 사용자 질문: '복지 포인트 백만 원 나온 거 어디서 쓸 수 있어?'
🚀 [FAISS 검색 엔진] 연관 지식 초고속 스캔 완료 (최고 코사인 유사도: 0.3816)
-----------------------------------------------------------------
📌 매칭 매뉴얼: 【복지 포인트 혜택】 (유사도: 0.3816)
💡 AI 핵심 지식 컨텍스트 제공:
매년 1월 1일 전 임직원에게 복지 포인트 100만 포인트가 지급되며, 사내 복지몰 또는 자기개발, 도서 구매, 건강검진 비용으로 사용할 수 있습니다.


사내 업무 매뉴얼 데이터를 TF-IDF 벡터라이저로 수치화하고, 이를 파이썬 import만으로 가동되는 초고속 벡터 검색 엔진인 FAISS에 주입하여 연동하는 로컬 RAG(검색) 확장 코드입니다.
FAISS는 기본적으로 유클리드 거리(IndexFlatL2) 외에도, 내부 벡터들을 미리 정규화(Normalization)해두면 코사인 유사도(IndexFlatIP, 내적 연산)를 초고속으로 계산할 수 있는 기능을 제공합니다. 한국어 조사 격파를 위한 글자 n-gram 옵션과 FAISS 전용 데이터 규격 변환을 결합하여 완벽한 파이프라인을 구축했습니다.

------------------------------

## 🧐 FAISS 확장 파이프라인의 핵심 전환 원리

* toarray().astype('float32'): scikit-learn 모델은 연산 최적화를 위해 메모리를 쥐어짜는 scipy 희소 행렬을 사용하지만, 페이스북의 FAISS 엔진은 C++ 기반의 로우 레벨 초고속 처리를 고집하기 때문에 오직 32비트 실수형 넘파이 배열 형태만 입력으로 수용합니다. 이 다리를 놓아주는 변환 작업이 필수입니다.
* faiss.normalize_L2() + IndexFlatIP: 코사인 유사도 공식은 수학적으로 두 벡터의 내적 값을 벡터의 크기(길이)로 나누는 연산입니다. 애초에 벡터들의 길이를 전부 1로 칼같이 스케일 정규화를 시켜두면, 무거운 나눗셈 연산 없이 오직 직선 곱하기 연산인 내적(IndexFlatIP) 연산만 수행해도 완벽한 코사인 유사도 값이 도출됩니다. FAISS가 0.0001초 만에 수백만 건을 검색해 내는 핵심 비결입니다.

외부 서버나 API 연동 비용을 단 1원도 쓰지 않고, 순수 파이썬 라이브러리 import 조합만으로 현대 생성형 AI 인프라의 핵심인 초고속 임베딩 벡터 검색 아키텍처를 완벽하게 설계하고 가동하셨습니다!

------------------------------

## TODO LIST

* [1. RAG 고도화] TF-IDF를 넘어서 단어의 진짜 문맥적 의미를 인지하는 OpenAI/허깅페이스의 임베딩(Embedding) 모델을 연동하여 지식 검색 고도화하기

* [2. 모델 영구화] 메모리가 꺼져도 검색이 유지되도록 방금 만든 FAISS 인덱스를 하드디스크에 파일로 내보내고 동기화하는 인덱스 저장 및 로드 실습 진행하기
